# Signature Verification System — Interactive Pipeline

Interactive **spatial-branch** workflow for the Multimodal Signature Verification
System (EE 499). The temporal branch and full multi-corpus training runs are driven
by `scripts/train.py` — see `docs/COLAB_RUNBOOK.md`. This notebook is for
inspection and visualization, not for producing the shipped weights.

1. **Environment Setup & VRAM Management**: Configures hardware acceleration (`cuda` / `cpu`) and executes `torch.cuda.empty_cache()` to prevent Out-Of-Memory errors.
2. **Dataset Discovery & Visualization**: Loads real corpora via `src.data.parsers` and displays sample signature pairs.
3. **Writer-Independent Split (ISO/IEC 23053)**: Partitions writers into Train (70%), Validation (15%), and Test (15%) sets without writer overlap.
4. **Conditional Training**: Checks for compiled `.pth` model weights. Trains the Siamese CNN with **early stopping** if weights are missing, saving loss curves inline.
5. **Comprehensive Evaluation**: Computes **Accuracy**, **Equal Error Rate (EER)**, **False Acceptance Rate (FAR)**, and **False Rejection Rate (FRR)**.
6. **Visualizations**: Plots **Confusion Matrix**, **ROC Curve**, **DET Curve**, and a **Sample Predictions Grid** showing probe signature pairs alongside predictions and confidence scores.

> **Pairing note:** negatives here are **skilled forgeries** — a genuine signature
> against a deliberate imitation of that same writer — per the AGENTS.md evaluation
> protocol. Random cross-writer impostors are trivially separable and would report
> an EER far better than the system's real-world performance.


In [ ]:
import os
import sys
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, det_curve

# Add project root directory to path
sys.path.insert(0, os.path.abspath('.'))

from config.settings import (
    SPATIAL_EMBEDDING_DIM, TEMPORAL_EMBEDDING_DIM,
    CONTRASTIVE_MARGIN, LEARNING_RATE, BATCH_SIZE,
    NUM_EPOCHS, WEIGHT_DECAY, MODEL_WEIGHTS_DIR,
    DECISION_THRESHOLD, TARGET_FAR, TARGET_FRR,
    HARD_NEGATIVE_RATIO, DATA_DIR,
    EARLY_STOPPING_PATIENCE, EARLY_STOPPING_MIN_DELTA,
)
from src.models.siamese import SpatialSiamese, TemporalSiamese, ContrastiveLoss, FusionEngine
from src.data.dataset import (
    preprocess_image, quality_gate, writer_independent_split,
    generate_skilled_pairs, SpatialSignatureDataset
)
from src.data.parsers import parse_dataset

# Random seeds for reproducible evaluation
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Dynamic hardware device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Device Selected: {device}")

# Clear GPU memory cache to prevent Out-Of-Memory (OOM) errors
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM memory cache cleared successfully.")


In [ ]:
# Load real corpora through the dataset parsers.
#
# The previous os.walk() discovery treated any directory containing images as a
# single "writer", which would have collapsed all 55 CEDAR signers into one
# entry (data/cedar/full_org) and silently destroyed writer independence. It
# also filtered on .png/.jpg only, skipping the .tif files BHSig260 and UTSig
# ship. The parsers handle each corpus's real on-disk layout instead.
SPATIAL_CORPORA = ["cedar", "bhsig260", "utsig"]

samples_by_writer = {}
for name in SPATIAL_CORPORA:
    path = os.path.join(DATA_DIR, name)
    if os.path.isdir(path):
        parsed = parse_dataset(path)
        samples_by_writer.update(parsed)
        print(f"{name}: {len(parsed)} writers")

print(f"Total: {len(samples_by_writer)} writers")

# Fallback: generate a small synthetic corpus so the notebook runs without data
if not samples_by_writer:
    print("No corpora found under ./data. Generating synthetic signature pairs for demonstration...")
    demo_root = os.path.join(DATA_DIR, "demo_dataset")
    os.makedirs(demo_root, exist_ok=True)

    for writer in range(1, 5):
        genuine, forged = [], []
        for i in range(1, 6):
            img = np.ones((256, 256), dtype=np.uint8) * 255
            cv2.line(img, (30 + i * 5, 128), (220, 128 + writer * 4), 0, 3)
            path = f"{demo_root}/w{writer}_g{i}.png"
            cv2.imwrite(path, img)
            genuine.append(path)

            img = np.ones((256, 256), dtype=np.uint8) * 255
            cv2.line(img, (34 + i * 5, 132), (216, 124 + writer * 4), 0, 3)
            path = f"{demo_root}/w{writer}_f{i}.png"
            cv2.imwrite(path, img)
            forged.append(path)

        samples_by_writer[f"demo_{writer}"] = {"genuine": genuine, "forged": forged}
    print(f"Demo dataset created with {len(samples_by_writer)} writers.")

# Generate sample pairs for preview visualization (skilled-forgery negatives)
demo_pairs, demo_labels = generate_skilled_pairs(
    samples_by_writer, num_genuine_per_writer=2, num_forged_per_writer=2)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Dataset Sample Preview — Genuine vs Skilled-Forgery Pairs", fontsize=14, fontweight="bold")

preview = list(zip(demo_pairs, demo_labels))
random.Random(42).shuffle(preview)

for ax, ((p1, p2), label) in zip(axes.flat, preview[:8]):
    img1 = preprocess_image(p1)[0]
    img2 = preprocess_image(p2)[0]
    ax.imshow(np.hstack([img1, img2]), cmap="gray")
    status = "Genuine Pair (y=1)" if label == 1 else "Skilled Forgery (y=0)"
    ax.set_title(status, color="green" if label == 1 else "red", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
os.makedirs("./results", exist_ok=True)
plt.savefig("./results/dataset_samples.png", dpi=150)
plt.show()


In [ ]:
# Perform ISO/IEC 23053 writer-independent data partition split
writer_ids = sorted(samples_by_writer.keys())
train_ids, val_ids, test_ids = writer_independent_split(writer_ids, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42)

train_samples = {w: samples_by_writer[w] for w in train_ids}
val_samples = {w: samples_by_writer[w] for w in val_ids} if val_ids else train_samples
test_samples = {w: samples_by_writer[w] for w in test_ids} if test_ids else train_samples

# Pairs are formed AFTER the split, so no writer straddles a partition boundary.
train_pairs, train_labels = generate_skilled_pairs(train_samples, seed=42)
val_pairs, val_labels = generate_skilled_pairs(val_samples, seed=43)
test_pairs, test_labels = generate_skilled_pairs(test_samples, seed=44)

print(f"Writer Split Ratios: {len(train_ids)} Train | {len(val_ids)} Val | {len(test_ids)} Test")
print(f"Generated Pairs: {len(train_pairs)} Train | {len(val_pairs)} Val | {len(test_pairs)} Test")

# Construct PyTorch DataLoaders
train_loader = DataLoader(SpatialSignatureDataset(train_pairs, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SpatialSignatureDataset(val_pairs, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SpatialSignatureDataset(test_pairs, test_labels), batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
weights_path = os.path.join(MODEL_WEIGHTS_DIR, "spatial_cnn.pth")
FORCE_TRAIN = False  # Set to True to force re-training

model = SpatialSiamese(embedding_dim=SPATIAL_EMBEDDING_DIM).to(device)
loss_fn = ContrastiveLoss(margin=CONTRASTIVE_MARGIN, hard_negative_ratio=HARD_NEGATIVE_RATIO)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

if os.path.exists(weights_path) and not FORCE_TRAIN:
    print(f"Pre-compiled model weights found at: {weights_path}")
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print("Weights loaded successfully into model. Skipping training phase.")
else:
    print(f"Starting training phase for {NUM_EPOCHS} epochs...")
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    
    for epoch in range(NUM_EPOCHS):
        # Clear GPU VRAM cache prior to each epoch to prevent OOM
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        model.train()
        t_loss = 0.0
        for x1, x2, labels in train_loader:
            x1, x2, labels = x1.to(device), x2.to(device), labels.to(device)
            optimizer.zero_grad()
            emb1, emb2 = model(x1, x2)
            loss = loss_fn(emb1, emb2, labels)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            
        t_loss /= max(len(train_loader), 1)
        
        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for x1, x2, labels in val_loader:
                x1, x2, labels = x1.to(device), x2.to(device), labels.to(device)
                emb1, emb2 = model(x1, x2)
                v_loss += loss_fn(emb1, emb2, labels).item()
        v_loss /= max(len(val_loader), 1)
        
        train_losses.append(t_loss)
        val_losses.append(v_loss)
        
        if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
            print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")
            
        # Early stopping: halt once validation loss stalls, and keep only the
        # best-validation weights so a late overfitting epoch cannot overwrite them.
        if v_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
            best_val_loss = v_loss
            epochs_without_improvement = 0
            os.makedirs(MODEL_WEIGHTS_DIR, exist_ok=True)
            torch.save(model.state_dict(), weights_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch+1} (no improvement for "
                      f"{EARLY_STOPPING_PATIENCE} epochs). Best val loss: {best_val_loss:.4f}")
                break
            
    print(f"Training complete. Best model weights saved to: {weights_path}")
    
    # Plot Training vs Validation Loss Curve
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss", linewidth=2)
    plt.plot(range(1, len(val_losses)+1), val_losses, label="Validation Loss", linewidth=2, linestyle="--")
    plt.xlabel("Epoch")
    plt.ylabel("Contrastive Loss")
    plt.title("Spatial CNN — Training vs Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("./results/training_loss.png", dpi=150)
    plt.show()

In [ ]:
# Clear GPU cache before running evaluation
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.eval()
genuine_distances, impostor_distances = [], []
all_labels, all_distances = [], []

with torch.no_grad():
    for x1, x2, labels in test_loader:
        x1, x2, labels = x1.to(device), x2.to(device), labels.to(device)
        emb1, emb2 = model(x1, x2)
        distances = torch.nn.functional.pairwise_distance(emb1, emb2, p=2).cpu().numpy()
        labels_np = labels.cpu().numpy()
        
        for dist, lbl in zip(distances, labels_np):
            all_distances.append(dist)
            all_labels.append(lbl)
            if lbl == 1:
                genuine_distances.append(dist)
            else:
                impostor_distances.append(dist)

all_distances = np.array(all_distances)
all_labels = np.array(all_labels)

# Convert L2 distance to similarity score: score = 1 / (1 + distance)
scores = 1.0 / (1.0 + all_distances)
genuine_scores = 1.0 / (1.0 + np.array(genuine_distances))
impostor_scores = 1.0 / (1.0 + np.array(impostor_distances))

# Calculate EER
fpr, tpr, thresholds = roc_curve(all_labels, scores)
fnr = 1.0 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0
eer_threshold = thresholds[eer_idx]

# FAR, FRR, Accuracy at decision threshold tau = 0.5
predictions = (scores >= DECISION_THRESHOLD).astype(int)
far = np.mean(impostor_scores >= DECISION_THRESHOLD)
frr = np.mean(genuine_scores < DECISION_THRESHOLD)
accuracy = np.mean(predictions == all_labels)

print("=" * 55)
print("             EVALUATION REPORT & METRICS")
print("=" * 55)
print(f"  Test Accuracy:                  {accuracy * 100:.2f}%")
print(f"  Equal Error Rate (EER):        {eer * 100:.2f}% (Threshold @ EER: {eer_threshold:.4f})")
print(f"  False Acceptance Rate (FAR):   {far * 100:.2f}% (Target: <{TARGET_FAR * 100:.0f}%)")
print(f"  False Rejection Rate (FRR):    {frr * 100:.2f}% (Target: <{TARGET_FRR * 100:.0f}%)")
print("=" * 55)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(all_labels, predictions)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Forged", "Genuine"])
axes[0].set_yticklabels(["Forged", "Genuine"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14, fontweight="bold")

fig.colorbar(im, ax=axes[0])

# 2. ROC Curve Plot
axes[1].plot(fpr, tpr, linewidth=2, label=f"ROC (EER={eer*100:.2f}%)")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].scatter([fpr[eer_idx]], [tpr[eer_idx]], color="red", s=50, zorder=5, label=f"EER Point ({eer*100:.1f}%)")
axes[1].set_xlabel("False Acceptance Rate (FAR)")
axes[1].set_ylabel("True Acceptance Rate (1-FRR)")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. DET Curve Plot
fpr_det, fnr_det, _ = det_curve(all_labels, scores)
axes[2].plot(fpr_det * 100, fnr_det * 100, linewidth=2, color="purple")
axes[2].set_xlabel("False Acceptance Rate (%)")
axes[2].set_ylabel("False Rejection Rate (%)")
axes[2].set_title("DET Curve")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./results/evaluation_plots.png", dpi=150)
plt.show()

# 4. Sample Predictions Grid (Probe Pairs with Predictions and Confidence)
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
fig.suptitle("Sample Model Predictions vs Ground Truth", fontsize=14, fontweight="bold")

test_pairs_sample = test_pairs[:8]
for idx, (ax, (p1, p2)) in enumerate(zip(axes.flat, test_pairs_sample)):
    img1 = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(p2, cv2.IMREAD_GRAYSCALE)
    combined = np.hstack([img1, img2])
    ax.imshow(combined, cmap="gray")
    
    gt = "Genuine" if test_labels[idx] == 1 else "Forged"
    pred = "Genuine" if predictions[idx] == 1 else "Forged"
    score_pct = min(max(scores[idx] * 100, 0.0), 100.0)
    
    color = "green" if gt == pred else "red"
    title_str = f"GT: {gt} | Pred: {pred}\nConf: {score_pct:.1f}%"
    ax.set_title(title_str, color=color, fontsize=9, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("./results/sample_predictions.png", dpi=150)
plt.show()

# Clear GPU VRAM cache after full notebook execution
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Pipeline execution complete. VRAM cache cleared.")